# Module 5: pandas II, cleaning messy data

**Goal:** turn a messy real-world table into a trustworthy one, and *record every decision* so others (and future you) can check it.

**You will learn:** spotting problems · text cleanup · numbers stored as text · dates · duplicates · missing values ·
outliers · merging tables · writing a reusable cleaning function

**Maths in this module:** percentages and ratios · percent change · logarithms (and why we use them) · the IQR outlier rule

> 🧹 Analysts spend a large share of their time on cleaning. Doing it carefully is a core skill, not a chore.
> **Golden rules:** (1) never edit the raw file, (2) do every fix in code, (3) check the data *before and after* each step.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv(DATA / "messy_orders.csv")
raw.head(8)

## 1. Diagnose first

Before fixing anything, list what's wrong. A short **audit** of any new table:

In [ ]:
print("shape:", raw.shape)
print("exact duplicate rows:", raw.duplicated().sum())
print()
print(raw.isna().sum())
print()
raw.info()

In [ ]:
# Look at the distinct values of categorical-looking columns
print(raw["city"].value_counts())

Problems we can already see in this coffee-shop dataset:

| column | problem |
|---|---|
| whole table | 12 duplicate rows |
| `city`, `item`, `customer` | same value written many ways (`Bangkok`, `bangkok`, `  Bangkok `, …) and a misspelling (`Chiangmai`) |
| `order_date` | stored as text in several formats |
| `unit_price` | text with `$` signs; some missing |
| `quantity` | some missing; some impossible values (99) |

> 💡 Notice that some prices are `NaN`. The CSV had the text `N/A` there, and pandas quietly turned it into a missing value.
> You can control this with `pd.read_csv(..., na_values=[...])`.

We'll work on a copy so `raw` stays untouched:

In [ ]:
df = raw.copy()

## 2. Duplicates

Exact duplicates usually come from a data-export or double-entry glitch. Drop them, but **count first**.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"removed {before - len(df)} duplicates ({(before - len(df)) / before * 100:.1f}% of rows)")

> Duplicates by *ID* can differ in other columns; use `df.duplicated(subset=["order_id"])` to find them,
> then decide which row to keep.

## 3. Text clean-up

The `.str` accessor applies string methods to a whole column. Standardise case and whitespace, then fix known spelling variants.

In [ ]:
for col in ["customer", "city", "item"]:
    df[col] = df[col].str.strip().str.title()

df["city"] = df["city"].replace({"Chiangmai": "Chiang Mai"})

print(df["city"].value_counts())
print(df["item"].value_counts())

## 4. Numbers stored as text

`unit_price` contains `"$4.50"`. Strip the symbol, then convert. `errors="coerce"` turns anything unparseable into `NaN`
instead of crashing, so *check how many you created*.

In [ ]:
df["unit_price"] = pd.to_numeric(df["unit_price"].str.replace("$", "", regex=False), errors="coerce")
print(df["unit_price"].dtype, "| missing:", df["unit_price"].isna().sum())
df["unit_price"].describe()

## 5. Dates

Dates arrived in three formats. `format="mixed"` tells pandas to parse each value on its own.

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")
print(df["order_date"].dtype, df["order_date"].min().date(), "->", df["order_date"].max().date())

> ⚠️ **Ambiguous dates:** is `03/04/2024` 3 April or March 4? pandas guesses, and guesses wrong sometimes. When you control the
> source, prefer unambiguous formats like `2024-04-03`. When you don't, pass an explicit `format="%d/%m/%Y"`.

## 6. Missing values: choices, not just fixes

Why is the value missing? The answer decides the fix.

| strategy | when it's reasonable | risk |
|---|---|---|
| **drop the rows** (`dropna`) | few rows, missing at random | loses data; can bias if missingness isn't random |
| **fill with a typical value** (median/mean) | numeric, a few missing | shrinks the spread artificially |
| **fill using related columns** (e.g. item's usual price) | strong relationship exists | more code; can be wrong |
| **keep it missing** and flag it | the absence *is* information | many functions need handling |

First measure the problem as a **percentage**:

In [ ]:
missing_pct = (df.isna().mean() * 100).round(1)
print(missing_pct[missing_pct > 0])

In [ ]:
# Price: the same item always has (nearly) the same price, so fill from the item's typical price.
item_price = df.groupby("item")["unit_price"].median()
print(item_price)

df["price_was_missing"] = df["unit_price"].isna()          # keep a flag of what we changed
df["unit_price"] = df["unit_price"].fillna(df["item"].map(item_price))
print("missing prices left:", df["unit_price"].isna().sum())

## 7. Outliers

An **outlier** is a value far from the rest: sometimes a real extreme, sometimes a typo. Look first:

In [ ]:
print(df["quantity"].value_counts().sort_index())

### 🧮 Maths link: the IQR rule

With quartiles $Q_1$ and $Q_3$ and $\text{IQR}=Q_3-Q_1$, flag values outside

$$[\,Q_1-1.5\,\text{IQR},\;\;Q_3+1.5\,\text{IQR}\,]$$

It is robust because quartiles aren't pulled around by the extreme values themselves (unlike the mean and standard deviation).

In [ ]:
q = df["quantity"].dropna()
q1, q3 = q.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr} -> keep values in [{low}, {high}]")

is_outlier = (df["quantity"] < low) | (df["quantity"] > high)
df.loc[is_outlier]

The rule only *flags* values; **you** decide. A quantity of 99 coffees in one order is surely a typo (probably 9 or 1),
so we treat it as missing and then fill with the median quantity. If those had been real big orders, deleting
them would have removed the most interesting rows!

In [ ]:
df.loc[is_outlier, "quantity"] = np.nan
df["quantity"] = df["quantity"].fillna(df["quantity"].median())
print(df["quantity"].value_counts().sort_index())

## 8. New columns, and the maths of ratios & percent change

Revenue is a product; a share is a ratio; growth is a percent change:

$$\text{share} = \frac{\text{part}}{\text{total}}\times100\qquad
\text{change}\,\% = \frac{\text{new}-\text{old}}{\text{old}}\times100$$

In [ ]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.to_period("M")

by_city = df.groupby("city")["revenue"].sum()
print((by_city / by_city.sum() * 100).round(1))        # share of revenue per city, in %

In [ ]:
monthly = df.groupby("month")["revenue"].sum()
print(monthly.round(0))
print((monthly.pct_change() * 100).round(1))           # month-on-month percent change

### 🧮 Maths link: logarithms

$\log$ turns **multiplying into adding**: $\log(a\cdot b)=\log a+\log b$. So a log scale shows *ratios* (a doubling is
always the same distance), which makes right-skewed data like prices, incomes or follower counts much easier to read.
A tiny example, with 1 → 10 → 100 → 1000 (each ×10):

In [ ]:
x = pd.Series([1, 10, 100, 1000])
print(np.log10(x).tolist())          # [0, 1, 2, 3]: equal steps for equal ratios
print(np.log(2))                     # natural log, base e
skewed = pd.Series([1, 2, 2, 3, 3, 4, 5, 8, 20, 400])
print("skewed mean/median:", skewed.mean(), skewed.median())
print("log   mean/median:", np.log10(skewed).mean().round(2), np.log10(skewed).median().round(2))

## 9. Combining tables: `merge`

Real projects use several tables. Join them on a shared **key**.

In [ ]:
catalog = pd.DataFrame({
    "item": ["Latte", "Espresso", "Mocha", "Tea", "Cake"],
    "category": ["coffee", "coffee", "coffee", "tea", "food"],
})

merged = df.merge(catalog, on="item", how="left")
print(len(df), "->", len(merged), "rows (should be unchanged for a many-to-one join)")
merged.groupby("category")["revenue"].sum().round(1)

> `how="left"` keeps every row of the left table. After any merge, **check the row count** and look for new `NaN`s:
> an unexpected jump in rows means the key wasn't unique in the right table.

## 10. Wrap it in a function: the step to a reusable pipeline

Everything above ran in a notebook, top to bottom. Pull the steps into one function that takes the raw table and returns the clean one.
This is exactly what will go into your project's **`clean.py`**.

In [ ]:
def clean_orders(raw: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy of the raw coffee-shop orders table."""
    df = raw.drop_duplicates().reset_index(drop=True)

    for col in ["customer", "city", "item"]:
        df[col] = df[col].str.strip().str.title()
    df["city"] = df["city"].replace({"Chiangmai": "Chiang Mai"})

    df["unit_price"] = pd.to_numeric(df["unit_price"].str.replace("$", "", regex=False), errors="coerce")
    df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")

    item_price = df.groupby("item")["unit_price"].median()
    df["unit_price"] = df["unit_price"].fillna(df["item"].map(item_price))

    # Quantities above 50 are typos (e.g. 99): treat as missing, then fill with the median.
    df.loc[df["quantity"] > 50, "quantity"] = np.nan
    df["quantity"] = df["quantity"].fillna(df["quantity"].median())

    df["revenue"] = df["quantity"] * df["unit_price"]
    return df


clean = clean_orders(raw)
print(clean.shape, "| total revenue:", clean["revenue"].sum())
clean.head()

Always finish with **sanity checks**, assertions that stop you if something silently went wrong:

In [ ]:
assert clean.duplicated().sum() == 0
assert clean[["quantity", "unit_price", "revenue"]].notna().all().all()
assert set(clean["city"]) == {"Bangkok", "Chiang Mai", "Phuket"}
assert clean["quantity"].between(1, 5).all()
print("✅ all checks passed")

## Recap

* Diagnose → fix → verify. Count what you change. Keep the raw file read-only.
* `.str` for text · `to_numeric` / `to_datetime` for types · `drop_duplicates` · `fillna` / `dropna` as *decisions*
* IQR rule flags outliers; you decide what they mean · percentages, ratios and logs describe change and scale
* Put the steps in a function with assertions: that's your project's `clean.py`

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** in the README.